# Testing LLMs for post-OCR error correction

This notebook, the second stage of the THISTLE pipeline, uses a large language model (LLM) via [Ollama](https://ollama.com) to correct OCR errors extracted in the previous step (`img_preprocessing.ipynb`).

## Why use a locally-run LLM rather than a chatbot like ChatGPT?

At first glance, prompting a chatbot to clean OCR text may seem equivalent to what this notebook does. There are several important differences, however:

- **Scale**: this notebook processes an entire dataset in a single run by iterating over every row in a `.csv` file. Manually copy-pasting a full dataset into a chatbot can be tedious, especially if the dataset is very large.
- **Reproducibility**: the ability to select a specific model version and `temperature` parameter, which controls how stochastic or 'creative' model outputs are, contributes to more consistent results and, in turn, greater reproducibility than working in a chatbot interface without these controls.
- **Model flexibility**: multiple open-access models can be tested and compared by changing a single variable (see Step 4). Chatbots are often closed-source; additionally, only certain LLMs are currently available as chatbots.
- **Temperature control**: For a task like OCR error correction, lower temperatures are preferable because they reduce the risk of the model generating additional information or taking creative license by, for instance, modernising historical language.
- **Privacy**: running models locally via Ollama means your data never leaves your machine. Similarly, if you are working on an HPC, your data will remain within the secure infrastructure of the HPC provider. Using a chatbot transfers input and output data to external servers of private companies, which can pose a significant GDPR risk.

## Models tested in this project

Three open-access models available through Ollama were tested in this project: `llama3`, `gemma2`, and `mistral`. All three models were run with `temperature=0.1` and the same prompt. Model performance was evaluated using WER and CER scores (see `accuracy_and_results.ipynb`).

To try a different model, change the `model_name` variable in Step 4 below. A full list of models available via Ollama can be found at [ollama.com/library](https://ollama.com/library).

## How preprocessed images feed into this workflow

The image preprocessing step (`img_preprocessing.ipynb`) applied contrast enhancement, noise reduction, and binarisation to each scanned image before passing it to Tesseract OCR. The purpose of this preprocessing was to produce cleaner, more accurate raw OCR text, which in turn gives the LLM better input to work with. This notebook loads that Tesseract output alongside the original ProQuest OCR and applies LLM-based correction to both, allowing a direct comparison of how much image preprocessing improved the starting quality of the text.

## For working in Colab

## Connect to the GPU:
In the 'Runtime' menu, click on 'Change runtime type' and select 'T4 GPU' under 'Hardware accelerator'.

**NOTE:** you will need to be logged in with your Google account to connect your Drive and to use the GPU. Free access is limited; for larger projects, you may need to consider working on an HPC.

## Connect your Google Drive:
Run the code block below to mount your Google Drive.

In [ ]:
#  you will need to approve the pop-up request to connect your Google Drive account
from google.colab import drive
drive.mount('/content/drive')

## Step 1: Install dependencies
Run this cell first to install all required Python packages.

- **In Google Colab:** packages are installed for your current session and will need reinstalling if you reconnect.
- **In Anaconda (local):** run this once per environment, or install via `requirements.txt` instead (see `code/README.md`).

Ollama must also be installed separately on your machine. See `code/README.md` for instructions.

In [ ]:
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install pandas ollama tqdm

## Step 2: Import packages and set up Ollama
The following code block imports all dependencies and starts the Ollama server.

In [ ]:
import pandas as pd
import ollama
from tqdm import tqdm
import time
import subprocess
import threading

def run_ollama_serve():
    subprocess.Popen(['ollama', 'serve'])

# start the Ollama server (required in Colab; if running locally, Ollama should already be running)
thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(15)
print('Ollama server started.')

## Step 3: Load data

This step loads two files:
- **`thistle_df`**: the ground truth CSV (`ground_truth.csv`), which contains manually transcribed articles from *The Scotsman* and their metadata, including the image filename (`png_img`) used to join the two datasets
- **`tesseract_df`**: the OCR output saved in the previous step (`processed_imgs.csv`), which contains the Tesseract OCR text extracted from the preprocessed images

The two are merged on the `png_img` column to produce a single dataframe containing both the ground truth and the OCR text for each article.

In [ ]:
# paste the path to ground_truth.csv (found in data/ in the cloned repo)
# example (Colab): "/content/drive/My Drive/your_project_folder/THISTLE_project/data/ground_truth.csv"
# example (local): "/Users/yourname/Documents/THISTLE_project/data/ground_truth.csv"
thistle_df = pd.read_csv('insert_your_path_to/ground_truth.csv')

# paste the path to the processed_imgs.csv saved in the previous step (img_preprocessing.ipynb)
# example (Colab): "/content/drive/My Drive/your_project_folder/processed_imgs.csv"
# example (local): "/Users/yourname/Documents/THISTLE_project/processed_imgs.csv"
tesseract_df = pd.read_csv('insert_your_path_to/processed_imgs.csv')


In [ ]:
# view the first 5 rows of the datasets
thistle_df.head()
tesseract_df.head()

In [ ]:
# merge the two dataframes on the image filename column
df = thistle_df.merge(tesseract_df, on='png_img', how='left')
df.head()

## Step 4: Select a model and define the correction function

This cell pulls a model from Ollama and defines a function that sends each row of OCR text to the model with a correction prompt.

### Choosing a model
Set `model_name` to the name of the Ollama model you want to use. The models tested in this project were:
- `'llama3'` — Meta's Llama 3 (8B); strong general-purpose instruction-following
- `'gemma2'` — Google DeepMind's Gemma 2 (9B); efficient, good on shorter texts
- `'mistral'` — Mistral AI's Mistral 7B; fast, but slightly more prone to modernising archaic language

To test multiple models, run this cell and the next with a different `model_name` each time, saving the output to a different column (e.g. `df['llama3_output']`, `df['gemma2_output']`). You can then compare WER and CER scores for each in `accuracy_and_results.ipynb`.

A full list of available models is at [ollama.com/library](https://ollama.com/library).

### Prompt design
The prompt instructs the model to correct OCR errors while preserving historical language. The key constraints are:
- correct misread characters, broken words, and garbled punctuation
- preserve archaic spellings and historical language conventions
- do not add any text not present in the original
- return only the corrected text, with no commentary

**Note:** You can experiment with the prompt wording, but take care not to instruct the model to 'improve' or 'rewrite' the text, as this wording seemed to increase the tendency to modernise the language or generate additional information.

In [ ]:
# set the model name — options tested in this project: 'llama3', 'gemma2', 'mistral'
model_name = 'llama3'
ollama.pull(model_name)

def correct_text(text):
    if pd.isna(text) or text.strip() == '':
        return ''

    prompt = (
        "Here is OCR-extracted text from a nineteenth-century newspaper in the public domain. "
        "Please correct OCR errors — such as misread characters, broken words, and garbled punctuation — "
        "so that the text makes grammatical and linguistic sense. "
        "Preserve the historical language and spelling conventions of the original; "
        "do not modernise archaic words or add any text that is not present in the original. "
        f"Return only the corrected text, with no additional commentary.\n\n{text}"
    )

    response = ollama.chat(
        model=model_name,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': 0.1}  # lower temperature = less stochastic (and lower risk of hallucination or over-correction)
    )
    return response['message']['content']

## Step 5: Run correction

This cell applies the `correct_text` function to two columns:
- `original_ocr`: the OCR text as supplied by ProQuest (without any image preprocessing)
- `tesseract_ocr`: the OCR text extracted by Tesseract from the preprocessed images

Corrected outputs are saved to new columns in the dataframe. Running this cell may take some time depending on the size of your dataset and the model selected. A progress bar is displayed via `tqdm`.

In [ ]:
# correct the original ProQuest OCR
df['corrected_original'] = [correct_text(t) for t in tqdm(df['original_ocr'])]

# correct the Tesseract OCR output from the preprocessing step
df['corrected_tesseract'] = [correct_text(t) for t in tqdm(df['tesseract_ocr'])]


## Step 6: Save the results

The output CSV is used as input in the next stage of the pipeline. If you are testing multiple models, save a separate CSV for each (e.g. `post_ocr_llama3.csv`, `post_ocr_gemma2.csv`) and update the column names in `accuracy_and_results.ipynb` accordingly.

In [ ]:
# paste the path where you would like to save the post-OCR output CSV
# example (Colab): "/content/drive/My Drive/your_project_folder/post_ocr_output.csv"
# example (local): "/Users/yourname/Documents/THISTLE_project/post_ocr_output.csv"
df.to_csv('insert_your_output_path/post_ocr_output.csv', index=False)
print('Saved successfully.')
